<a href="https://colab.research.google.com/github/stutinarain/CrackVision-Deployment-Aware-Crack-Segmentation-Benchmark/blob/main/code_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dataset source
This notebook downloads the dataset from the CrackForest GitHub repository and uses its `groundTruth` annotations, including `.mat` decoding support used in community discussions of the dataset. [web:40][web:71][web:68]


In [ ]:
!pip install -q segmentation-models-pytorch timm scikit-image opencv-python pandas matplotlib scipy

import os
import json
import time
import math
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
from skimage.morphology import skeletonize
from skimage.segmentation import find_boundaries

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
import torchvision
import segmentation_models_pytorch as smp

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## Download CrackForest from GitHub
This cell clones the dataset repository directly into Colab so the dataset path is fixed and reproducible.


In [ ]:
%cd /content
if not Path('/content/CrackForest-dataset').exists():
    !git clone https://github.com/cuilimeng/CrackForest-dataset.git
else:
    print('CrackForest-dataset already exists')

!find /content/CrackForest-dataset -maxdepth 3 | head -n 200


## Detect dataset folders inside the cloned repo
The code checks common folder names used in the CrackForest repository and pairs masks to images by filename stem.


In [ ]:
REPO_ROOT = Path('/content/CrackForest-dataset')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
MASK_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.mat'}
CANDIDATE_IMAGE_DIR_NAMES = ['image', 'images', 'Images', 'img', 'crackImages', 'crackimages']
CANDIDATE_MASK_DIR_NAMES = ['groundTruth', 'groundtruth', 'GroundTruth', 'mask', 'masks', 'gt', 'GT']

image_dirs = [p for p in REPO_ROOT.rglob('*') if p.is_dir() and p.name in CANDIDATE_IMAGE_DIR_NAMES]
mask_dirs = [p for p in REPO_ROOT.rglob('*') if p.is_dir() and p.name in CANDIDATE_MASK_DIR_NAMES]

print('Image dir candidates:')
for p in image_dirs:
    print(' ', p)
print('Mask dir candidates:')
for p in mask_dirs:
    print(' ', p)

def score_dir_pair(img_dir, mask_dir):
    img_files = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    mask_files = [p for p in mask_dir.iterdir() if p.is_file() and p.suffix.lower() in MASK_EXTS]
    img_stems = {p.stem for p in img_files}
    mask_stems = {p.stem for p in mask_files}
    return len(img_stems & mask_stems)

best_pair = None
best_overlap = -1
for i_dir in image_dirs:
    for m_dir in mask_dirs:
        ov = score_dir_pair(i_dir, m_dir)
        if ov > best_overlap:
            best_overlap = ov
            best_pair = (i_dir, m_dir)

if best_pair is None or best_overlap == 0:
    raise RuntimeError('Could not find matching image/mask folders inside the cloned CrackForest repo.')

IMAGE_DIR, MASK_DIR = best_pair
OUTPUT_DIR = Path('/content/final_crackforest_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Selected IMAGE_DIR:', IMAGE_DIR)
print('Selected MASK_DIR :', MASK_DIR)
print('Matched stems     :', best_overlap)


In [ ]:
def build_paired_files(image_dir, mask_dir):
    image_map = {p.stem: p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}
    mask_map = {p.stem: p for p in mask_dir.iterdir() if p.is_file() and p.suffix.lower() in MASK_EXTS}
    common = sorted(set(image_map.keys()) & set(mask_map.keys()))
    return [(str(image_map[k]), str(mask_map[k])) for k in common]

pairs = build_paired_files(IMAGE_DIR, MASK_DIR)
print('Total paired samples:', len(pairs))
print('Example pairs:')
for p in pairs[:5]:
    print(p)

if len(pairs) == 0:
    raise RuntimeError('No image-mask pairs found after cloning the dataset repo.')


## Hyperparameters and 3-fold setup
These are fully visible so the training loop is explicit.


In [ ]:
PATCH_SIZE = 480
BATCH_SIZE = 4
EPOCHS = 25
PATIENCE = 5
NUM_CLASSES = 2
LEARNING_RATE = 1e-4
NUM_WORKERS = 0

print('PATCH_SIZE   =', PATCH_SIZE)
print('BATCH_SIZE   =', BATCH_SIZE)
print('EPOCHS       =', EPOCHS)
print('PATIENCE     =', PATIENCE)
print('NUM_CLASSES  =', NUM_CLASSES)
print('LEARNING_RATE=', LEARNING_RATE)

def make_kfold_splits(n, k=3, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    splits = []
    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        splits.append((train_idx.tolist(), val_idx.tolist()))
    return splits

splits = make_kfold_splits(len(pairs), k=3, seed=SEED)
for i, (tr, va) in enumerate(splits, 1):
    print(f'Fold {i}: train={len(tr)} | val={len(va)}')


## Mask loader and severity mapping
This supports `.mat` masks and creates severity visualizations for figure generation.


In [ ]:
def load_crackforest_mask(mask_path):
    mask_path = Path(mask_path)
    if mask_path.suffix.lower() == '.mat':
        mat = sio.loadmat(str(mask_path))
        if 'groundTruth' in mat:
            arr = np.array(mat['groundTruth'][0][0][0])
            out = np.zeros_like(arr, dtype=np.uint8)
            out[arr == 2] = 255
            out[arr == 1] = 0
            return (out > 127).astype(np.uint8)
        raise ValueError(f'Unsupported .mat structure: {mask_path}')
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(mask_path)
    return (mask > 127).astype(np.uint8)

def create_severity_map(binary_mask):
    mask_u8 = (binary_mask > 0).astype(np.uint8)
    dist = cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return np.zeros_like(mask_u8, dtype=np.uint8)
    norm = dist / dist.max()
    severity = np.zeros_like(mask_u8, dtype=np.uint8)
    severity[(norm > 0) & (norm <= 0.33)] = 1
    severity[(norm > 0.33) & (norm <= 0.66)] = 2
    severity[(norm > 0.66)] = 3
    return severity


In [ ]:
class CrackForestDataset(Dataset):
    def __init__(self, pairs, indices, train=True, patch_size=480):
        self.samples = [pairs[i] for i in indices]
        self.train = train
        self.patch_size = patch_size

    def __len__(self):
        return len(self.samples)

    def _read_image(self, path):
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def _augment(self, img, mask):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2])
            mask = torch.flip(mask, dims=[1])
        if random.random() < 0.5:
            img = torch.flip(img, dims=[1])
            mask = torch.flip(mask, dims=[0])
        if random.random() < 0.4:
            angle = random.uniform(-15, 15)
            img = TF.rotate(img, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask.unsqueeze(0).float(), angle, interpolation=InterpolationMode.NEAREST).squeeze(0).long()
        return img, mask

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = self._read_image(img_path)
        mask = load_crackforest_mask(mask_path)
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()
        img = TF.resize(img, [self.patch_size, self.patch_size], interpolation=InterpolationMode.BILINEAR, antialias=True)
        mask = TF.resize(mask.unsqueeze(0), [self.patch_size, self.patch_size], interpolation=InterpolationMode.NEAREST).squeeze(0).long()
        if self.train:
            img, mask = self._augment(img, mask)
        img = TF.normalize(img, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return img, mask


## Dataset sanity check
Run this first to confirm the repo was downloaded and masks decode correctly.


In [ ]:
def denormalize(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (img_tensor.cpu() * std + mean).clamp(0, 1)

preview_ds = CrackForestDataset(pairs, list(range(min(3, len(pairs)))), train=False, patch_size=PATCH_SIZE)
fig, axes = plt.subplots(len(preview_ds), 3, figsize=(10, 4 * len(preview_ds)))
if len(preview_ds) == 1:
    axes = np.expand_dims(axes, 0)
for i in range(len(preview_ds)):
    img, mask = preview_ds[i]
    severity = create_severity_map(mask.numpy())
    axes[i, 0].imshow(denormalize(img).permute(1, 2, 0))
    axes[i, 0].set_title('Crack image')
    axes[i, 1].imshow(mask.numpy(), cmap='gray')
    axes[i, 1].set_title('Ground truth mask')
    axes[i, 2].imshow(severity, cmap='jet')
    axes[i, 2].set_title('Severity mapping')
    for j in range(3):
        axes[i, j].axis('off')
plt.tight_layout()
plt.show()


## Model definitions


In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, d=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, dilation=d, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class MultiScaleContextBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 4
        self.b1 = ConvBNAct(in_ch, mid, k=1, p=0)
        self.b2 = ConvBNAct(in_ch, mid, k=3, p=1, d=1)
        self.b3 = ConvBNAct(in_ch, mid, k=3, p=2, d=2)
        self.b4 = ConvBNAct(in_ch, mid, k=3, p=3, d=3)
        self.fuse = ConvBNAct(out_ch, out_ch, k=3, p=1)
    def forward(self, x):
        return self.fuse(torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1))

class SpatialChannelGate(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Conv2d(ch, ch // reduction, 1), nn.ReLU(inplace=True), nn.Conv2d(ch // reduction, ch, 1), nn.Sigmoid())
        self.spatial = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = x * self.fc(self.avg(x))
        avg_map = x.mean(dim=1, keepdim=True)
        max_map, _ = x.max(dim=1, keepdim=True)
        return x * self.spatial(torch.cat([avg_map, max_map], dim=1))

class EdgeGuidedRefinement(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.edge_head = nn.Sequential(ConvBNAct(ch, ch), nn.Conv2d(ch, 1, 1))
        self.refine = nn.Sequential(ConvBNAct(ch + 1, ch), ConvBNAct(ch, ch))
    def forward(self, x):
        edge_logits = self.edge_head(x)
        refined = self.refine(torch.cat([x, torch.sigmoid(edge_logits)], dim=1))
        return refined, edge_logits

class DecoderStage(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv1 = ConvBNAct(out_ch + skip_ch, out_ch)
        self.conv2 = ConvBNAct(out_ch, out_ch)
        self.attn = SpatialChannelGate(out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = self.conv1(torch.cat([x, skip], dim=1))
        x = self.conv2(x)
        return self.attn(x)

class ScaleTopoCrackNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        backbone = torchvision.models.resnet34(weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1 = backbone.layer1
        self.enc2 = backbone.layer2
        self.enc3 = backbone.layer3
        self.enc4 = backbone.layer4
        self.ms1 = MultiScaleContextBlock(64, 64)
        self.ms2 = MultiScaleContextBlock(64, 64)
        self.ms3 = MultiScaleContextBlock(128, 128)
        self.ms4 = MultiScaleContextBlock(256, 256)
        self.ms5 = MultiScaleContextBlock(512, 512)
        self.dec4 = DecoderStage(512, 256, 256)
        self.dec3 = DecoderStage(256, 128, 128)
        self.dec2 = DecoderStage(128, 64, 64)
        self.dec1 = DecoderStage(64, 64, 64)
        self.edge_refine = EdgeGuidedRefinement(64)
        self.seg_head = nn.Conv2d(64, num_classes, 1)
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.pool(x0)
        e1 = self.ms2(self.enc1(x1))
        e2 = self.ms3(self.enc2(e1))
        e3 = self.ms4(self.enc3(e2))
        e4 = self.ms5(self.enc4(e3))
        s0 = self.ms1(x0)
        d4 = self.dec4(e4, e3)
        d3 = self.dec3(d4, e2)
        d2 = self.dec2(d3, e1)
        d1 = self.dec1(d2, s0)
        refined, edge_logits = self.edge_refine(d1)
        seg_logits = self.seg_head(refined)
        seg_logits = F.interpolate(seg_logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        edge_logits = F.interpolate(edge_logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return seg_logits, edge_logits

class UNetBaseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=num_classes)
    def forward(self, x):
        seg = self.model(x)
        edge = torch.zeros((x.shape[0], 1, x.shape[2], x.shape[3]), device=x.device)
        return seg, edge

class AttentionUNetBaseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = smp.UnetPlusPlus(encoder_name='resnet34', encoder_weights='imagenet', in_channels=3, classes=num_classes)
    def forward(self, x):
        seg = self.model(x)
        edge = torch.zeros((x.shape[0], 1, x.shape[2], x.shape[3]), device=x.device)
        return seg, edge

class DeepLabV3PlusBaseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.model = smp.DeepLabV3Plus(encoder_name='resnet50', encoder_weights='imagenet', in_channels=3, classes=num_classes)
    def forward(self, x):
        seg = self.model(x)
        edge = torch.zeros((x.shape[0], 1, x.shape[2], x.shape[3]), device=x.device)
        return seg, edge


In [ ]:
def mask_to_boundary(mask_np):
    return find_boundaries(mask_np.astype(bool), mode='outer').astype(np.float32)

class ScaleTopoLoss(nn.Module):
    def __init__(self, lambda_dice=0.5, lambda_topo=0.3, lambda_edge=0.2):
        super().__init__()
        self.lambda_dice = lambda_dice
        self.lambda_topo = lambda_topo
        self.lambda_edge = lambda_edge
        self.ce = nn.CrossEntropyLoss()
        lap = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=torch.float32)
        self.register_buffer('lap_kernel', lap.view(1,1,3,3))
    def dice_loss(self, logits, target):
        prob = torch.softmax(logits, dim=1)[:,1]
        tgt = (target == 1).float()
        inter = (prob * tgt).sum(dim=(1,2))
        union = prob.sum(dim=(1,2)) + tgt.sum(dim=(1,2))
        dice = (2*inter + 1) / (union + 1)
        return 1 - dice.mean()
    def topology_loss(self, logits, target):
        prob = torch.softmax(logits, dim=1)[:,1:2]
        tgt = (target == 1).float().unsqueeze(1)
        pred_lap = F.conv2d(prob, self.lap_kernel.to(prob.device), padding=1)
        tgt_lap = F.conv2d(tgt, self.lap_kernel.to(prob.device), padding=1)
        return F.mse_loss(pred_lap, tgt_lap)
    def edge_loss(self, edge_logits, target):
        with torch.no_grad():
            edge_targets = []
            for t in target.detach().cpu().numpy():
                edge_targets.append(mask_to_boundary(t))
            edge_targets = torch.from_numpy(np.stack(edge_targets)).float().unsqueeze(1).to(edge_logits.device)
        return F.binary_cross_entropy_with_logits(edge_logits, edge_targets)
    def forward(self, seg_logits, edge_logits, target):
        ce = self.ce(seg_logits, target)
        dice = self.dice_loss(seg_logits, target)
        topo = self.topology_loss(seg_logits, target)
        edge = self.edge_loss(edge_logits, target)
        return ce + self.lambda_dice * dice + self.lambda_topo * topo + self.lambda_edge * edge

def binary_metrics(pred, target):
    pred = pred.astype(np.uint8)
    target = target.astype(np.uint8)
    tp = ((pred == 1) & (target == 1)).sum()
    tn = ((pred == 0) & (target == 0)).sum()
    fp = ((pred == 1) & (target == 0)).sum()
    fn = ((pred == 0) & (target == 1)).sum()
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    iou = tp / max(tp + fp + fn, 1)
    return float(acc), float(prec), float(rec), float(f1), float(iou)

def boundary_f1(pred, target):
    pb = find_boundaries(pred.astype(bool), mode='outer').astype(np.uint8)
    tb = find_boundaries(target.astype(bool), mode='outer').astype(np.uint8)
    tp = ((pb == 1) & (tb == 1)).sum()
    fp = ((pb == 1) & (tb == 0)).sum()
    fn = ((pb == 0) & (tb == 1)).sum()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return float(2 * prec * rec / max(prec + rec, 1e-8))

def skeleton_f1(pred, target):
    ps = skeletonize(pred.astype(bool)).astype(np.uint8)
    ts = skeletonize(target.astype(bool)).astype(np.uint8)
    tp = ((ps == 1) & (ts == 1)).sum()
    fp = ((ps == 1) & (ts == 0)).sum()
    fn = ((ps == 0) & (ts == 1)).sum()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return float(2 * prec * rec / max(prec + rec, 1e-8))

def all_segmentation_metrics(pred, target):
    acc, prec, rec, f1, iou = binary_metrics(pred, target)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'iou': iou, 'boundary_f1': boundary_f1(pred, target), 'skeleton_f1': skeleton_f1(pred, target)}

class EDSCalculator:
    def score(self, clean_iou, fps, params_M, memory_MB, noisy_iou, skeleton_f1_score, boundary_f1_score):
        fps_term = min(fps / 30.0, 1.0)
        size_term = 1.0 / (1.0 + params_M / 10.0)
        mem_term = 1.0 / (1.0 + memory_MB / 1000.0)
        eds = 0.25 * clean_iou + 0.15 * noisy_iou + 0.15 * skeleton_f1_score + 0.10 * boundary_f1_score + 0.15 * fps_term + 0.10 * size_term + 0.10 * mem_term
        return {'EDS': float(eds), 'fps_term': float(fps_term), 'size_term': float(size_term), 'mem_term': float(mem_term)}


In [ ]:
def add_noise_batch(x):
    return torch.clamp(x + torch.randn_like(x) * 0.04, -3.0, 3.0)

def count_params_and_memory(model):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return float(params / 1e6), float(params * 4 / (1024 ** 2))

@torch.no_grad()
def benchmark_fps(model, size=(1,3,480,480), warmup=5, runs=20):
    model.eval()
    x = torch.randn(*size).to(device)
    for _ in range(warmup):
        _ = model(x)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        _ = model(x)
    if device == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return float(runs / max(t1 - t0, 1e-8))

class Trainer:
    def __init__(self, model, train_loader, val_loader, model_name, epochs=25, lr=1e-4, patience=5):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.model_name = model_name
        self.epochs = epochs
        self.patience = patience
        self.criterion = ScaleTopoLoss()
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=epochs, eta_min=1e-6)
    def train_one_epoch(self):
        self.model.train()
        losses = []
        for imgs, masks in self.train_loader:
            imgs = imgs.to(device)
            masks = masks.to(device)
            self.opt.zero_grad()
            seg_logits, edge_logits = self.model(imgs)
            if seg_logits.shape[-2:] != masks.shape[-2:]:
                seg_logits = F.interpolate(seg_logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            if edge_logits.shape[-2:] != masks.shape[-2:]:
                edge_logits = F.interpolate(edge_logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = self.criterion(seg_logits, edge_logits, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
            losses.append(loss.item())
        self.sched.step()
        return float(np.mean(losses))
    @torch.no_grad()
    def evaluate(self, noisy=False):
        self.model.eval()
        scores = []
        for imgs, masks in self.val_loader:
            imgs = imgs.to(device)
            if noisy:
                imgs = add_noise_batch(imgs)
            seg_logits, edge_logits = self.model(imgs)
            preds = seg_logits.argmax(1).cpu().numpy()
            gts = masks.numpy()
            for p, g in zip(preds, gts):
                scores.append(all_segmentation_metrics(p, g))
        return {k: float(np.mean([s[k] for s in scores])) for k in scores[0].keys()}
    def fit(self, save_path):
        best_f1 = -1
        best_epoch = -1
        patience_counter = 0
        history = {'train_loss': [], 'val_f1': [], 'val_iou': [], 'val_boundary_f1': [], 'val_skeleton_f1': []}
        print(f'Training {self.model_name}')
        for epoch in range(1, self.epochs + 1):
            train_loss = self.train_one_epoch()
            val_metrics = self.evaluate(noisy=False)
            history['train_loss'].append(train_loss)
            history['val_f1'].append(val_metrics['f1'])
            history['val_iou'].append(val_metrics['iou'])
            history['val_boundary_f1'].append(val_metrics['boundary_f1'])
            history['val_skeleton_f1'].append(val_metrics['skeleton_f1'])
            print(f'Epoch {epoch:02d}/{self.epochs} | loss={train_loss:.4f} | f1={val_metrics["f1"]:.4f} | iou={val_metrics["iou"]:.4f} | boundary_f1={val_metrics["boundary_f1"]:.4f} | skeleton_f1={val_metrics["skeleton_f1"]:.4f}')
            if val_metrics['f1'] > best_f1:
                best_f1 = val_metrics['f1']
                best_epoch = epoch
                patience_counter = 0
                torch.save(self.model.state_dict(), save_path)
                print('  -> best updated')
            else:
                patience_counter += 1
                print(f'  -> no improvement ({patience_counter}/{self.patience})')
                if patience_counter >= self.patience:
                    print('  -> early stopping')
                    break
        self.model.load_state_dict(torch.load(save_path, map_location=device))
        print('Loaded best checkpoint from epoch', best_epoch)
        return history


## Save crack, mask, prediction, and severity images
Each fold saves model-specific visual panels for paper figures.


In [ ]:
def save_prediction_panels(model, model_name, fold_idx, val_ds, max_items=3):
    model.eval()
    rows = min(max_items, len(val_ds))
    fig, axes = plt.subplots(rows, 4, figsize=(14, 4 * rows))
    if rows == 1:
        axes = np.expand_dims(axes, 0)
    with torch.no_grad():
        for i in range(rows):
            img, mask = val_ds[i]
            seg_logits, edge_logits = model(img.unsqueeze(0).to(device))
            pred = seg_logits.argmax(1).squeeze(0).cpu().numpy()
            severity = create_severity_map(mask.numpy())
            axes[i, 0].imshow(denormalize(img).permute(1, 2, 0))
            axes[i, 0].set_title('Crack image')
            axes[i, 1].imshow(mask.numpy(), cmap='gray')
            axes[i, 1].set_title('Ground truth mask')
            axes[i, 2].imshow(pred, cmap='gray')
            axes[i, 2].set_title(f'{model_name} prediction')
            axes[i, 3].imshow(severity, cmap='jet')
            axes[i, 3].set_title('Severity mapping')
            for j in range(4):
                axes[i, j].axis('off')
    plt.tight_layout()
    out_path = OUTPUT_DIR / f'{model_name}_fold{fold_idx}_visuals.png'
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    return str(out_path)


## Explicit 3-fold benchmark runner
This shows the full 3-fold loop and epoch-based training clearly.


In [ ]:
def run_model_benchmark(model_name, model_ctor):
    all_fold_results = []
    print('' + '=' * 100)
    print('RUNNING MODEL:', model_name)
    print('=' * 100)
    for fold_idx, (train_idx, val_idx) in enumerate(splits, 1):
        print('' + '-' * 100)
        print(f'{model_name} | Fold {fold_idx}/3')
        print('-' * 100)
        train_ds = CrackForestDataset(pairs, train_idx, train=True, patch_size=PATCH_SIZE)
        val_ds = CrackForestDataset(pairs, val_idx, train=False, patch_size=PATCH_SIZE)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        model = model_ctor(num_classes=NUM_CLASSES)
        trainer = Trainer(model, train_loader, val_loader, model_name=f'{model_name}_fold{fold_idx}', epochs=EPOCHS, lr=LEARNING_RATE, patience=PATIENCE)
        ckpt_path = OUTPUT_DIR / f'{model_name}_fold{fold_idx}.pt'
        history = trainer.fit(ckpt_path)
        clean = trainer.evaluate(noisy=False)
        noisy = trainer.evaluate(noisy=True)
        fps = benchmark_fps(trainer.model, size=(1,3,PATCH_SIZE,PATCH_SIZE), warmup=5, runs=20)
        params_M, memory_MB = count_params_and_memory(trainer.model)
        eds = EDSCalculator().score(clean['iou'], fps, params_M, memory_MB, noisy['iou'], clean['skeleton_f1'], clean['boundary_f1'])
        visual_path = save_prediction_panels(trainer.model, model_name, fold_idx, val_ds, max_items=3)
        fold_result = {'model': model_name, 'fold': fold_idx, 'clean': clean, 'noisy': noisy, 'fps': fps, 'params_M': params_M, 'memory_MB': memory_MB, 'eds': eds, 'history': history, 'visual_path': visual_path}
        all_fold_results.append(fold_result)
        print(json.dumps({k: v for k, v in fold_result.items() if k != 'history'}, indent=2))
    with open(OUTPUT_DIR / f'{model_name}_results.json', 'w') as f:
        json.dump(all_fold_results, f, indent=2)
    return all_fold_results


## Separate training cell: ScaleTopoCrackNet


In [ ]:
ScaleTopoCrackNet_results = run_model_benchmark('ScaleTopoCrackNet', ScaleTopoCrackNet)


## Separate training cell: U-Net


In [ ]:
UNet_results = run_model_benchmark('UNet', UNetBaseline)


## Separate training cell: Attention U-Net / U-Net++


In [ ]:
AttentionUNet_results = run_model_benchmark('AttentionUNet', AttentionUNetBaseline)


## Separate training cell: DeepLabV3+


In [ ]:
DeepLabV3Plus_results = run_model_benchmark('DeepLabV3Plus', DeepLabV3PlusBaseline)


## Final aggregation cell


In [ ]:
combined_results = []
for var_name in ['ScaleTopoCrackNet_results', 'UNet_results', 'AttentionUNet_results', 'DeepLabV3Plus_results']:
    if var_name in globals():
        combined_results.extend(globals()[var_name])

if len(combined_results) == 0:
    print('Run at least one model training cell first.')
else:
    rows = []
    for model_name in sorted(set([r['model'] for r in combined_results])):
        model_rows = [r for r in combined_results if r['model'] == model_name]
        rows.append({
            'model': model_name,
            'accuracy': np.mean([r['clean']['accuracy'] for r in model_rows]),
            'precision': np.mean([r['clean']['precision'] for r in model_rows]),
            'recall': np.mean([r['clean']['recall'] for r in model_rows]),
            'f1': np.mean([r['clean']['f1'] for r in model_rows]),
            'iou': np.mean([r['clean']['iou'] for r in model_rows]),
            'boundary_f1': np.mean([r['clean']['boundary_f1'] for r in model_rows]),
            'skeleton_f1': np.mean([r['clean']['skeleton_f1'] for r in model_rows]),
            'noisy_iou': np.mean([r['noisy']['iou'] for r in model_rows]),
            'fps': np.mean([r['fps'] for r in model_rows]),
            'params_M': np.mean([r['params_M'] for r in model_rows]),
            'memory_MB': np.mean([r['memory_MB'] for r in model_rows]),
            'EDS': np.mean([r['eds']['EDS'] for r in model_rows])
        })
    summary_df = pd.DataFrame(rows).sort_values('f1', ascending=False)
    display(summary_df.round(4))
    summary_df.to_csv(OUTPUT_DIR / 'final_summary.csv', index=False)
    print('Saved final summary to', OUTPUT_DIR / 'final_summary.csv')


In [ ]:
from pathlib import Path
import inspect
import torch

OUTPUTDIR = Path("/content/final_crackforest_outputs")
print("OUTPUTDIR exists:", OUTPUTDIR.exists())
print("PT files:", [p.name for p in OUTPUTDIR.glob("*.pt")][:20])

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

print("ScaleTopoCrackNet init:", inspect.signature(ScaleTopoCrackNet.__init__))
print("UNetBaseline init:", inspect.signature(UNetBaseline.__init__))
print("AttentionUNetBaseline init:", inspect.signature(AttentionUNetBaseline.__init__))
print("DeepLabV3PlusBaseline init:", inspect.signature(DeepLabV3PlusBaseline.__init__))

In [ ]:
!pip install -q onnx onnxruntime onnxscript

from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import inspect

OUTPUTDIR = Path("/content/final_crackforest_outputs")
EXPORT_DIR = OUTPUTDIR / "onnx_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
PATCHSIZE = 480

print("Step 1: setup done")

def build_scaletopo():
    sig = str(inspect.signature(ScaleTopoCrackNet.__init__))
    print("ScaleTopo signature:", sig)
    if "num_classes" in sig:
        return ScaleTopoCrackNet(num_classes=2)
    return ScaleTopoCrackNet()

print("Step 2: finding checkpoint")
cands = list(OUTPUTDIR.glob("*ScaleTopoCrackNet*fold1*.pt")) + list(OUTPUTDIR.glob("*ScaleTopoCrackNet*_fold1*.pt"))
print("Candidates:", [c.name for c in cands])

if not cands:
    raise FileNotFoundError("No ScaleTopoCrackNet fold1 checkpoint found")

ckpt_path = cands[0]
print("Using checkpoint:", ckpt_path)

print("Step 3: building model")
model = build_scaletopo().to(device)

print("Step 4: loading state dict")
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state)
model.eval()

print("Step 5: creating dummy input")
dummy = torch.randn(1, 3, PATCHSIZE, PATCHSIZE, device=device)

onnx_path = EXPORT_DIR / "ScaleTopoCrackNet_fold1.onnx"
print("Step 6: exporting to", onnx_path)

with torch.no_grad():
    torch.onnx.export(
        model,
        dummy,
        str(onnx_path),
        export_params=True,
        opset_version=18,  # Changed opset_version to 18 as suggested by the warning
        do_constant_folding=True,
        input_names=["image"],
        output_names=["seg", "edge"],
        dynamic_axes={
            "image": {0: "batch_size"},
            "seg": {0: "batch_size"},
            "edge": {0: "batch_size"}
        },
        verbose=False
    )

print("Step 7: checking ONNX file")
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("Step 8: running ONNX Runtime")
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
x = np.random.randn(1, 3, PATCHSIZE, PATCHSIZE).astype(np.float32)
outs = session.run(None, {"image": x})

print("Done")
print("seg shape:", outs[0].shape)
print("edge shape:", outs[1].shape)
print("Saved to:", onnx_path)

In [ ]:
# ===== FULL ONNX EXPORT CELL FOR ALL 4 MODELS =====

!pip install -q onnx onnxruntime

from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort

OUTPUTDIR = Path("/content/final_crackforest_outputs")
OUTPUTDIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = OUTPUTDIR / "onnx_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
PATCHSIZE = 480
NUMCLASSES = 2

print("Using device:", device)
print("Checkpoint dir:", OUTPUTDIR)
print("Export dir:", EXPORT_DIR)

def build_model_by_name(model_name):
    # Ensure models are defined globally or imported correctly
    if model_name == "ScaleTopoCrackNet":
        return ScaleTopoCrackNet(num_classes=NUMCLASSES)
    elif model_name == "UNet":
        return UNetBaseline(num_classes=NUMCLASSES)
    elif model_name == "AttentionUNet":
        return AttentionUNetBaseline(num_classes=NUMCLASSES)
    elif model_name == "DeepLabV3Plus":
        return DeepLabV3PlusBaseline(num_classes=NUMCLASSES)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

def find_checkpoint(model_name, fold_idx=1):
    candidates = [
        OUTPUTDIR / f"{model_name}_fold{fold_idx}.pt",
        OUTPUTDIR / f"{model_name}fold{fold_idx}.pt",
    ]

    for ckpt in candidates:
        if ckpt.exists():
            return ckpt

    matches = sorted(OUTPUTDIR.glob(f"*{model_name}*fold{fold_idx}*.pt"))
    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(f"No checkpoint found for {model_name} fold {fold_idx}")

def export_one_model(model_name, fold_idx=1):
    print("\n" + "=" * 80)
    print(f"Exporting {model_name} | fold {fold_idx}")
    print("=" * 80)

    ckpt_path = find_checkpoint(model_name, fold_idx)
    print("Checkpoint found:", ckpt_path)

    model = build_model_by_name(model_name).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Model loaded successfully")

    dummy_input = torch.randn(1, 3, PATCHSIZE, PATCHSIZE, device=device)
    onnx_path = EXPORT_DIR / f"{model_name}_fold{fold_idx}.onnx"

    print("Starting export...")
    with torch.no_grad():
        torch.onnx.export(
            model,
            dummy_input,
            str(onnx_path),
            export_params=True,
            opset_version=18, # Use opset_version 18 as recommended by PyTorch
            do_constant_folding=True,
            input_names=["image"],
            output_names=["seg", "edge"],
            dynamic_axes={
                "image": {0: "batch_size"},
                "seg": {0: "batch_size"},
                "edge": {0: "batch_size"},
            },
            verbose=False,
        )

    print("Export complete:", onnx_path)

    print("Checking ONNX model...")
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print("ONNX checker passed")

    print("Running ONNX Runtime verification...")
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    x = np.random.randn(1, 3, PATCHSIZE, PATCHSIZE).astype(np.float32)
    outputs = session.run(None, {"image": x})

    print("ONNX Runtime passed")
    print("seg shape :", outputs[0].shape)
    print("edge shape:", outputs[1].shape)

    return str(onnx_path)

model_names = [
    "ScaleTopoCrackNet",
    "UNet",
    "AttentionUNet",
    "DeepLabV3Plus",
]

onnx_files = []

for fold_idx in [1, 2, 3]:
    for model_name in model_names:
        try:
            exported_path = export_one_model(model_name, fold_idx=fold_idx)
            onnx_files.append(exported_path)
            print(f"SUCCESS: {model_name} fold {fold_idx}") # Added fold_idx for clarity
        except Exception as e:
            print(f"FAILED: {model_name} fold {fold_idx} -> {e}") # Added fold_idx for clarity

print("\n" + "=" * 80)
print("FINAL EXPORTED ONNX FILES")
print("=" * 80)

if len(onnx_files) == 0:
    print("No ONNX files were exported.")
else:
    for f in onnx_files:
        print(f)


In [ ]:
from pathlib import Path
import onnxruntime as ort

EXPORT_DIR = Path("/content/final_crackforest_outputs/onnx_exports")
onnx_files = list(EXPORT_DIR.glob("*.onnx"))

print(f"Found {len(onnx_files)} ONNX files")
for f in onnx_files:
    session = ort.InferenceSession(str(f), providers=["CPUExecutionProvider"])
    print(f"✓ {f.name} OK (inputs: {len(session.get_inputs())}, outputs: {len(session.get_outputs())})")

In [ ]:
!zip -r crackforest_onnx_models.zip /content/final_crackforest_outputs/onnx_exports/

In [ ]:
# =========================
# FULL SIMULATION + BENCHMARK + DOCS + PACKAGE PIPELINE
# =========================

!pip install -q onnxruntime opencv-python matplotlib pandas numpy pillow pyyaml

import os
import io
import json
import time
import math
import shutil
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort

In [ ]:
# -------------------------------------------------
# PATHS
# -------------------------------------------------
OUTPUTDIR = Path("/content/final_crackforest_outputs")
ONNX_DIR = OUTPUTDIR / "onnx_exports"
SIM_DIR = OUTPUTDIR / "deployment_simulation"
DOCS_DIR = SIM_DIR / "docs"
ARTIFACTS_DIR = SIM_DIR / "artifacts"
VIS_DIR = SIM_DIR / "visuals"
BENCH_DIR = SIM_DIR / "benchmarks"
PKG_DIR = SIM_DIR / "package"

for d in [SIM_DIR, DOCS_DIR, ARTIFACTS_DIR, VIS_DIR, BENCH_DIR, PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [ ]:
# -------------------------------------------------
# DATASET PATHS
# -------------------------------------------------
REPOROOT = Path("/content/CrackForest-dataset")
PATCHSIZE = 480
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

IMAGEEXTS = [".jpg", ".jpeg", ".png", ".bmp"]
MASKEXTS = [".png", ".jpg", ".jpeg", ".bmp", ".mat"]

CANDIDATE_IMAGE_DIRNAMES = ["image", "images", "Images", "img", "crackImages", "crackimages"]
CANDIDATE_MASK_DIRNAMES = ["groundTruth", "groundtruth", "GroundTruth", "mask", "masks", "gt", "GT"]


In [ ]:
# -------------------------------------------------
# HELPERS
# -------------------------------------------------
def find_best_image_mask_dirs(repo_root):
    imagedirs = [p for p in repo_root.rglob("*") if p.is_dir() and p.name in CANDIDATE_IMAGE_DIRNAMES]
    maskdirs = [p for p in repo_root.rglob("*") if p.is_dir() and p.name in CANDIDATE_MASK_DIRNAMES]

    def score_pair(imgdir, maskdir):
        imgfiles = [p for p in imgdir.iterdir() if p.is_file() and p.suffix.lower() in IMAGEEXTS]
        maskfiles = [p for p in maskdir.iterdir() if p.is_file() and p.suffix.lower() in MASKEXTS]
        imgstems = {p.stem for p in imgfiles}
        maskstems = {p.stem for p in maskfiles}
        return len(imgstems & maskstems)

    bestpair = None
    bestoverlap = -1
    for idir in imagedirs:
        for mdir in maskdirs:
            ov = score_pair(idir, mdir)
            if ov > bestoverlap:
                bestoverlap = ov
                bestpair = (idir, mdir)

    if bestpair is None or bestoverlap <= 0:
        raise RuntimeError("Could not find matching image/mask folders.")

    return bestpair

def build_paired_files(imagedir, maskdir):
    imagemap = {p.stem: p for p in imagedir.iterdir() if p.is_file() and p.suffix.lower() in IMAGEEXTS}
    maskmap = {p.stem: p for p in maskdir.iterdir() if p.is_file() and p.suffix.lower() in MASKEXTS}
    common = sorted(set(imagemap.keys()) & set(maskmap.keys()))
    return [(imagemap[k], maskmap[k]) for k in common]

def load_mask(mask_path):
    mask_path = Path(mask_path)
    if mask_path.suffix.lower() == ".mat":
        import scipy.io as sio
        mat = sio.loadmat(str(mask_path))
        if "groundTruth" in mat:
            arr = np.array(mat["groundTruth"][0][0][0], dtype=np.uint8)
            out = np.zeros_like(arr, dtype=np.uint8)
            out[arr >= 2] = 255
            out[arr < 2] = 0
            return (out > 127).astype(np.uint8)
        raise ValueError(f"Unsupported .mat structure: {mask_path}")
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise FileNotFoundError(mask_path)
    return (mask > 127).astype(np.uint8)

def load_rgb(image_path):
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_image(img_rgb, patchsize=480):
    original_h, original_w = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (patchsize, patchsize), interpolation=cv2.INTER_LINEAR)
    x = resized.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    x = np.transpose(x, (2, 0, 1))
    x = np.expand_dims(x, axis=0).astype(np.float32)
    return x, resized, (original_h, original_w)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def postprocess_outputs(seg_logits, edge_logits, original_size):
    seg_prob = softmax_np(seg_logits, axis=1)[0, 1]
    seg_mask = np.argmax(seg_logits, axis=1)[0].astype(np.uint8)
    edge_prob = 1.0 / (1.0 + np.exp(-edge_logits[0, 0]))

    h, w = original_size
    seg_prob_up = cv2.resize(seg_prob, (w, h), interpolation=cv2.INTER_LINEAR)
    seg_mask_up = cv2.resize(seg_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    edge_prob_up = cv2.resize(edge_prob, (w, h), interpolation=cv2.INTER_LINEAR)
    return seg_prob_up, seg_mask_up, edge_prob_up

def overlay_mask(image_rgb, mask, color=(255, 0, 0), alpha=0.35):
    out = image_rgb.copy()
    color_arr = np.array(color, dtype=np.uint8)
    idx = mask > 0
    out[idx] = ((1 - alpha) * out[idx] + alpha * color_arr).astype(np.uint8)
    return out

def binary_metrics(pred, target):
    pred = pred.astype(np.uint8)
    target = target.astype(np.uint8)
    tp = np.sum((pred == 1) & (target == 1))
    tn = np.sum((pred == 0) & (target == 0))
    fp = np.sum((pred == 1) & (target == 0))
    fn = np.sum((pred == 0) & (target == 1))

    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    iou = tp / max(tp + fp + fn, 1)
    return {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "iou": float(iou),
    }

def measure_session_latency(session, input_name, sample_input, warmup=5, runs=30):
    for _ in range(warmup):
        _ = session.run(None, {input_name: sample_input})
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = session.run(None, {input_name: sample_input})
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000.0)
    times = np.array(times)
    return {
        "latency_ms_mean": float(times.mean()),
        "latency_ms_std": float(times.std()),
        "latency_ms_p50": float(np.percentile(times, 50)),
        "latency_ms_p90": float(np.percentile(times, 90)),
        "latency_ms_p95": float(np.percentile(times, 95)),
        "fps_estimate": float(1000.0 / max(times.mean(), 1e-8)),
    }

In [ ]:
# -------------------------------------------------
# FIND DATA
# -------------------------------------------------
IMAGEDIR, MASKDIR = find_best_image_mask_dirs(REPOROOT)
pairs = build_paired_files(IMAGEDIR, MASKDIR)
if len(pairs) == 0:
    raise RuntimeError("No paired images/masks found.")

print("IMAGEDIR:", IMAGEDIR)
print("MASKDIR :", MASKDIR)
print("Total pairs:", len(pairs))


In [ ]:
# -------------------------------------------------
# DISCOVER ONNX FILES
# -------------------------------------------------
onnx_files = sorted(ONNX_DIR.glob("*.onnx"))
if len(onnx_files) == 0:
    raise RuntimeError(f"No ONNX files found in {ONNX_DIR}")

print("Found ONNX files:", len(onnx_files))
for f in onnx_files:
    print(" -", f.name)


In [ ]:
# -------------------------------------------------
# SELECT SAMPLE SET FOR SIMULATION
# -------------------------------------------------
NUM_SAMPLES = min(12, len(pairs))
sample_pairs = pairs[:NUM_SAMPLES]

In [ ]:
# -------------------------------------------------
# RUN SIMULATION + BENCHMARK
# -------------------------------------------------
all_rows = []

for onnx_path in onnx_files:
    model_name = onnx_path.stem
    model_vis_dir = VIS_DIR / model_name
    model_vis_dir.mkdir(parents=True, exist_ok=True)

    print("\nRunning model:", model_name)
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name

    # benchmark using first sample
    first_img = load_rgb(sample_pairs[0][0])
    sample_input, _, _ = preprocess_image(first_img, patchsize=PATCHSIZE)
    bench = measure_session_latency(session, input_name, sample_input, warmup=5, runs=30)

    onnx_size_mb = onnx_path.stat().st_size / (1024 * 1024)

    for idx, (img_path, mask_path) in enumerate(sample_pairs, start=1):
        img_rgb = load_rgb(img_path)
        gt_mask = load_mask(mask_path)

        x, _, original_size = preprocess_image(img_rgb, patchsize=PATCHSIZE)
        outputs = session.run(None, {input_name: x})

        seg_logits = outputs[0]
        edge_logits = outputs[1]

        seg_prob, pred_mask, edge_prob = postprocess_outputs(seg_logits, edge_logits, original_size)
        overlay = overlay_mask(img_rgb, pred_mask)

        metrics = binary_metrics(pred_mask, gt_mask)

        stem = Path(img_path).stem
        base = model_vis_dir / stem

        cv2.imwrite(str(base.with_name(base.name + "_input.png")), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(base.with_name(base.name + "_gtmask.png")), (gt_mask * 255).astype(np.uint8))
        cv2.imwrite(str(base.with_name(base.name + "_predmask.png")), (pred_mask * 255).astype(np.uint8))
        cv2.imwrite(str(base.with_name(base.name + "_segprob.png")), (seg_prob * 255).clip(0,255).astype(np.uint8))
        cv2.imwrite(str(base.with_name(base.name + "_edgeprob.png")), (edge_prob * 255).clip(0,255).astype(np.uint8))
        cv2.imwrite(str(base.with_name(base.name + "_overlay.png")), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        all_rows.append({
            "model": model_name,
            "image_id": stem,
            "onnx_file": str(onnx_path),
            "onnx_size_mb": onnx_size_mb,
            "latency_ms_mean": bench["latency_ms_mean"],
            "latency_ms_std": bench["latency_ms_std"],
            "latency_ms_p50": bench["latency_ms_p50"],
            "latency_ms_p90": bench["latency_ms_p90"],
            "latency_ms_p95": bench["latency_ms_p95"],
            "fps_estimate": bench["fps_estimate"],
            **metrics
        })


In [ ]:
# -------------------------------------------------
# SAVE BENCHMARK TABLES
# -------------------------------------------------
results_df = pd.DataFrame(all_rows)
results_csv = BENCH_DIR / "simulation_per_image_results.csv"
results_df.to_csv(results_csv, index=False)

summary_df = (
    results_df
    .groupby("model", as_index=False)
    .agg({
        "onnx_size_mb": "mean",
        "latency_ms_mean": "mean",
        "latency_ms_std": "mean",
        "latency_ms_p50": "mean",
        "latency_ms_p90": "mean",
        "latency_ms_p95": "mean",
        "fps_estimate": "mean",
        "accuracy": "mean",
        "precision": "mean",
        "recall": "mean",
        "f1": "mean",
        "iou": "mean",
    })
    .sort_values("f1", ascending=False)
)

summary_csv = BENCH_DIR / "simulation_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nSaved:")
print(results_csv)
print(summary_csv)

In [ ]:
# -------------------------------------------------
# CREATE COMPARISON CHARTS
# -------------------------------------------------
plt.figure(figsize=(12, 5))
plt.bar(summary_df["model"], summary_df["f1"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean F1")
plt.title("Model Comparison - Mean F1")
plt.tight_layout()
plt.savefig(BENCH_DIR / "chart_f1.png", dpi=200)
plt.close()

plt.figure(figsize=(12, 5))
plt.bar(summary_df["model"], summary_df["iou"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean IoU")
plt.title("Model Comparison - Mean IoU")
plt.tight_layout()
plt.savefig(BENCH_DIR / "chart_iou.png", dpi=200)
plt.close()

plt.figure(figsize=(12, 5))
plt.bar(summary_df["model"], summary_df["latency_ms_mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Latency (ms)")
plt.title("Model Comparison - Mean ONNX CPU Latency")
plt.tight_layout()
plt.savefig(BENCH_DIR / "chart_latency.png", dpi=200)
plt.close()

plt.figure(figsize=(12, 5))
plt.bar(summary_df["model"], summary_df["onnx_size_mb"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Model Size (MB)")
plt.title("Model Comparison - ONNX Size")
plt.tight_layout()
plt.savefig(BENCH_DIR / "chart_size.png", dpi=200)
plt.close()

In [ ]:
# -------------------------------------------------
# PICK A RECOMMENDED MODEL
# simple heuristic: best f1, then iou, then latency
# -------------------------------------------------
summary_df = summary_df.sort_values(
    by=["f1", "iou", "latency_ms_mean"],
    ascending=[False, False, True]
).reset_index(drop=True)

recommended_model = summary_df.iloc[0]["model"]
recommended_row = summary_df.iloc[0].to_dict()


In [ ]:
# -------------------------------------------------
# GENERATE DEPLOYMENT DOCUMENTATION
# -------------------------------------------------
readme_text = f"""
# Crack Segmentation Deployment Simulation Package

This package was generated from the CrackForest ONNX export workflow.

## What is included
- ONNX models exported from the trained notebook
- Inference simulation results on sample dataset images
- Per-image visual outputs including masks, probabilities, and overlays
- Benchmark summaries for latency, FPS estimate, model size, and segmentation quality
- A recommended deployment candidate based on simulation metrics

## Input/Output contract
- Input tensor name: image
- Input shape used in simulation: (1, 3, 480, 480)
- Output 1: seg logits, expected shape approximately (1, 2, 480, 480)
- Output 2: edge logits, expected shape approximately (1, 1, 480, 480)

## Preprocessing
- Resize image to 480x480
- Convert RGB to float32 in [0,1]
- Normalize with:
  - mean = [0.485, 0.456, 0.406]
  - std = [0.229, 0.224, 0.225]

## Postprocessing
- Apply softmax over segmentation logits
- Use argmax to obtain binary crack mask
- Apply sigmoid over edge logits for edge probability map
- Resize prediction outputs back to original image size

## Recommended candidate
- Model: {recommended_model}

## Recommendation rationale
This recommendation is based on the simulation summary using mean F1, mean IoU, and latency trade-offs from the generated ONNX inference runs.

## Notes
This is a simulation-ready deployment package, not a hardware deployment. It is appropriate for:
- thesis/demo submission
- handoff to a deployment engineer
- later conversion to TensorRT / OpenVINO / edge runtime
"""

deployment_json = {
    "project": "CrackForest ONNX Deployment Simulation",
    "input_name": "image",
    "input_shape": [1, 3, 480, 480],
    "outputs": [
        {"name": "seg", "shape": [1, 2, 480, 480]},
        {"name": "edge", "shape": [1, 1, 480, 480]}
    ],
    "preprocessing": {
        "resize": [480, 480],
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std": [0.229, 0.224, 0.225],
        "color_format": "RGB",
        "dtype": "float32"
    },
    "postprocessing": {
        "segmentation": "softmax + argmax",
        "edge": "sigmoid",
        "resize_back_to_original": True
    },
    "recommended_model": recommended_model,
    "recommended_metrics": recommended_row,
    "artifacts": {
        "summary_csv": str(summary_csv),
        "per_image_csv": str(results_csv),
        "visuals_dir": str(VIS_DIR),
        "onnx_dir": str(ONNX_DIR)
    }
}

next_steps_md = f"""
# Next Steps

## Immediate
1. Use `{recommended_model}` as the primary simulated deployment candidate.
2. Use `simulation_summary.csv` in your report/presentation.
3. Use visuals from `deployment_simulation/visuals/` to show qualitative performance.

## If you later get hardware
1. Convert the recommended ONNX file to TensorRT or another hardware-specific runtime.
2. Benchmark on-device latency and memory.
3. Validate the same preprocessing and postprocessing contract.
4. Package as a lightweight API or embedded service.

## For thesis / project report
- Include ONNX export success
- Include benchmark charts
- Include qualitative prediction panels
- Include deployment-ready pipeline documentation
"""

with open(DOCS_DIR / "README.md", "w") as f:
    f.write(readme_text.strip() + "\n")

with open(DOCS_DIR / "deployment_spec.json", "w") as f:
    json.dump(deployment_json, f, indent=2)

with open(DOCS_DIR / "next_steps.md", "w") as f:
    f.write(next_steps_md.strip() + "\n")


In [ ]:
# -------------------------------------------------
# COPY KEY FILES INTO PACKAGE FOLDER
# -------------------------------------------------
shutil.copy2(results_csv, PKG_DIR / results_csv.name)
shutil.copy2(summary_csv, PKG_DIR / summary_csv.name)
shutil.copy2(DOCS_DIR / "README.md", PKG_DIR / "README.md")
shutil.copy2(DOCS_DIR / "deployment_spec.json", PKG_DIR / "deployment_spec.json")
shutil.copy2(DOCS_DIR / "next_steps.md", PKG_DIR / "next_steps.md")

for chart_name in ["chart_f1.png", "chart_iou.png", "chart_latency.png", "chart_size.png"]:
    src = BENCH_DIR / chart_name
    if src.exists():
        shutil.copy2(src, PKG_DIR / chart_name)

# copy top-level onnx files
PKG_ONNX_DIR = PKG_DIR / "onnx_models"
PKG_ONNX_DIR.mkdir(parents=True, exist_ok=True)
for f in onnx_files:
    shutil.copy2(f, PKG_ONNX_DIR / f.name)

# copy a few visuals from recommended model
PKG_VIS_DIR = PKG_DIR / "sample_visuals"
PKG_VIS_DIR.mkdir(parents=True, exist_ok=True)
recommended_vis_dir = VIS_DIR / recommended_model
sample_visuals = sorted(recommended_vis_dir.glob("*_overlay.png"))[:5]
for f in sample_visuals:
    stem = f.name.replace("_overlay.png", "")
    for suffix in ["_input.png", "_gtmask.png", "_predmask.png", "_segprob.png", "_edgeprob.png", "_overlay.png"]:
        src = recommended_vis_dir / f"{stem}{suffix}"
        if src.exists():
            shutil.copy2(src, PKG_VIS_DIR / src.name)


In [ ]:
# -------------------------------------------------
# ZIP PACKAGE
# -------------------------------------------------
zip_path = OUTPUTDIR / "deployment_simulation_package.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PKG_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PKG_DIR))

print("\n" + "="*80)
print("SIMULATION PACKAGE READY")
print("="*80)
print("Recommended model:", recommended_model)
print("Summary CSV:", summary_csv)
print("Per-image CSV:", results_csv)
print("Docs dir:", DOCS_DIR)
print("Visuals dir:", VIS_DIR)
print("Package dir:", PKG_DIR)
print("ZIP package:", zip_path)

print("\nTop models by simulation:")
display(summary_df.round(4))